# Lab 04 — Evaluation metrics

**Course:** Fundamentals of NLP
**Author:** Aymen Ben Brik

## Objectives

1. Build a confusion matrix and compute Accuracy, Precision, Recall, F1 from scratch.
2. Demonstrate the class-imbalance pitfall of accuracy.
3. Plot ROC and Precision-Recall curves.
4. Implement Perplexity for a small language model.
5. Implement BLEU and ROUGE on translation/summary pairs.

## Prerequisites

```bash
pip install scikit-learn matplotlib nltk rouge-score
```

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_curve, auc,
                             precision_recall_curve, average_precision_score)

## 2. Build a binary classifier (sci.space vs comp.graphics)

In [ ]:
categories = ['sci.space', 'comp.graphics']
data = fetch_20newsgroups(subset='all', categories=categories,
                           remove=('headers', 'footers', 'quotes'))
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42, stratify=data.target,
)
vec = TfidfVectorizer(min_df=5, max_df=0.7, stop_words='english')
Xtr, Xte = vec.fit_transform(X_train), vec.transform(X_test)
clf = LogisticRegression(max_iter=2000).fit(Xtr, y_train)
y_pred = clf.predict(Xte)
y_score = clf.predict_proba(Xte)[:, 1]   # probability of class 1

## 3. Confusion matrix and metrics from scratch

In [ ]:
def metrics_from_scratch(y_true, y_pred):
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                Accuracy=accuracy, Precision=precision, Recall=recall, F1=f1)

scratch = metrics_from_scratch(np.asarray(y_test), np.asarray(y_pred))
for k, v in scratch.items():
    print(f'  {k:<10} = {v:.3f}' if isinstance(v, float) else f'  {k:<10} = {v}')

## 4. Compare with sklearn

In [ ]:
print(classification_report(y_test, y_pred, target_names=categories, digits=3))

## 5. Class-imbalance pitfall

In [ ]:
# Build an artificial imbalanced test where 95% of inputs are class 0
rng = np.random.default_rng(42)
idx_pos = np.where(y_test == 1)[0]; idx_neg = np.where(y_test == 0)[0]
n_pos = max(1, len(idx_pos) // 10)
imb_idx = np.concatenate([idx_neg, rng.choice(idx_pos, size=n_pos, replace=False)])
y_imb = np.asarray(y_test)[imb_idx]; y_pred_imb = np.asarray(y_pred)[imb_idx]
print(f'Class 1 prevalence in imbalanced test: {y_imb.mean():.2%}')

# Naive baseline that always predicts 0
from sklearn.metrics import accuracy_score
print('Always-zero accuracy:', accuracy_score(y_imb, np.zeros_like(y_imb)))
print('Real classifier accuracy:', accuracy_score(y_imb, y_pred_imb))
print('\nAlways-zero classification report:')
print(classification_report(y_imb, np.zeros_like(y_imb), target_names=categories, zero_division=0))

**Observation.** The trivial baseline reaches a high *accuracy* but its precision and recall on the positive class are zero. **Always inspect class-conditional metrics under imbalance.**

## 6. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
ap = average_precision_score(y_test, y_score)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f'ROC (AUC={roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curve'); axes[0].legend()

axes[1].plot(recall_curve, precision_curve, label=f'PR (AP={ap:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve'); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Perplexity of a unigram and bigram language model

In [ ]:
import math
from collections import Counter

# Tiny corpus
train = 'the cat sat on the mat the cat ate the fish the dog ran on the floor'.split()
test = 'the cat ran on the floor'.split()
V = set(train); N_test = len(test)

# Unigram model with add-1 smoothing
uni = Counter(train); total = sum(uni.values())
def P_uni(w):
    return (uni[w] + 1) / (total + len(V))

log_prob_uni = sum(math.log2(P_uni(w)) for w in test)
ppl_uni = 2 ** (-log_prob_uni / N_test)
print(f'Unigram perplexity on test: {ppl_uni:.3f}')

# Bigram model with add-1 smoothing
bigrams = Counter(zip(train, train[1:]))
def P_bi(w_next, w_prev):
    return (bigrams[(w_prev, w_next)] + 1) / (uni[w_prev] + len(V))

log_prob_bi = math.log2(P_uni(test[0]))
for prev, w in zip(test, test[1:]):
    log_prob_bi += math.log2(P_bi(w, prev))
ppl_bi = 2 ** (-log_prob_bi / N_test)
print(f'Bigram perplexity on test:  {ppl_bi:.3f}')

**Observation.** The bigram model typically achieves a lower perplexity than the unigram, because conditioning on the previous word improves prediction.

## 8. BLEU score

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = 'Tunisia won the African Cup of Nations'.split()
candidates = [
    'Tunisia won the cup'.split(),                       # very short
    'Tunisia African the won Cup of Nations'.split(),    # right words, wrong order
    'Tunisia won the African Cup of Nations'.split(),    # exact match
]
smoothing = SmoothingFunction().method1
for c in candidates:
    bleu = sentence_bleu([reference], c, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    print(f'BLEU = {bleu:.3f}  for: {" ".join(c)}')

## 9. ROUGE score

In [ ]:
try:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    for c in candidates:
        scores = scorer.score(' '.join(reference), ' '.join(c))
        print(f'\nCandidate: {" ".join(c)}')
        for k, v in scores.items():
            print(f'  {k}: P={v.precision:.3f}  R={v.recall:.3f}  F={v.fmeasure:.3f}')
except ImportError:
    print('Install rouge-score: pip install rouge-score')

## Exercises

**Exercise 1.** A NER tagger finds 80 true entities, misses 20, and produces 30 spurious entities. Compute Precision, Recall, F1 by hand and verify with `precision_recall_fscore_support`.

**Exercise 2.** Build a fully imbalanced classification problem (e.g., 99% class 0). Train a real classifier and compare its F1 to that of a naive always-zero baseline. Plot the PR curve.

**Exercise 3.** Implement Perplexity from scratch on a held-out passage of the Brown corpus, using the bigram model trained on the rest of the corpus.

**Exercise 4.** Implement a simplified BLEU-1 (unigram precision + brevity penalty) without using NLTK. Verify on `reference = 'the cat is on the mat'` and `candidate = 'the cat sits on the mat'`.

**Exercise 5 (synthesis).** Design an evaluation protocol for a Tunisian-Arabic dialect-to-French translation system. Identify which metrics you would report, which biases each carries (e.g., BLEU's surface sensitivity), and how you would complement them with human evaluation.